In [ ]:
# 02_preprocesamiento.ipynb
# Preprocesamiento y curacion de datos
# Seleccion de variables, limpieza de nulos, eliminacion de duplicados,
# tratamiento de outliers con transformacion logaritmica

import os
import pandas as pd
import numpy as np

# Configuracion
RUTA_RAW = "./../data/raw/universal_top_spotify_songs.csv"
RUTA_PROCESSED = "./../data/processed/"

# Crear directorio de salida si no existe
os.makedirs(RUTA_PROCESSED, exist_ok=True)

# ============================================
# 1. CARGA DEL DATASET
# ============================================
print("Cargando dataset...")
df = pd.read_csv(RUTA_RAW)
print(f"Dataset cargado. Dimensiones: {df.shape}")
print(f"Columnas totales: {len(df.columns)}")

# ============================================
# 2. SELECCION DE VARIABLES
# ============================================
# Variables acusticas seleccionadas para clustering
FEATURES_ACUSTICAS = [
    'danceability',      # Rango 0-1: que tan bailable es
    'energy',            # Rango 0-1: intensidad y actividad
    'valence',           # Rango 0-1: positividad o alegria
    'acousticness',      # Rango 0-1: que tan acustica es
    'instrumentalness',  # Rango 0-1: ausencia de voz
    'liveness',          # Rango 0-1: presencia de audiencia en vivo
    'speechiness',       # Rango 0-1: cantidad de palabras habladas
    'tempo',             # BPM: velocidad de la cancion
    'loudness',          # dB: volumen (valores negativos)
    'duration_ms',       # milisegundos: duracion
    'mode'               # 0 o 1: tono menor o mayor
]

# Verificar que todas las columnas existen
columnas_faltantes = [col for col in FEATURES_ACUSTICAS if col not in df.columns]
if columnas_faltantes:
    raise ValueError(f"Columnas no encontradas: {columnas_faltantes}")

print("Variables seleccionadas para clustering:")
for i, col in enumerate(FEATURES_ACUSTICAS, 1):
    print(f"  {i}. {col}")

# ============================================
# 3. DIAGNOSTICO DE CALIDAD
# ============================================
print("\n" + "="*50)
print("DIAGNOSTICO DE CALIDAD")
print("="*50)

# 3.1 Valores nulos
nulos = df[FEATURES_ACUSTICAS].isnull().sum()
total_nulos = nulos.sum()
print(f"Valores nulos en features acusticas: {total_nulos}")
if total_nulos > 0:
    print(nulos[nulos > 0])

# 3.2 Registros duplicados
duplicados = df.duplicated().sum()
print(f"Registros duplicados (filas identicas): {duplicados}")

# 3.3 Verificacion de rangos
print("\nVerificacion de rangos:")
for col in FEATURES_ACUSTICAS:
    if col == 'mode':
        fuera_rango = df[(df[col] != 0) & (df[col] != 1)]
    elif col == 'tempo':
        fuera_rango = df[df[col] <= 0]
    elif col == 'duration_ms':
        fuera_rango = df[df[col] <= 0]
    else:
        fuera_rango = df[(df[col] < 0) | (df[col] > 1)]
    
    if len(fuera_rango) > 0:
        print(f"  {col}: {len(fuera_rango)} valores fuera de rango")

# ============================================
# 4. LIMPIEZA DE DATOS
# ============================================
print("\n" + "="*50)
print("APLICANDO LIMPIEZA")
print("="*50)

df_limpio = df.copy()
registros_iniciales = len(df_limpio)

# 4.1 Eliminar valores nulos
df_limpio = df_limpio.dropna(subset=FEATURES_ACUSTICAS)
print(f"Paso 1 - Eliminar nulos: {registros_iniciales - len(df_limpio)} registros eliminados")

# 4.2 Eliminar duplicados exactos
df_limpio = df_limpio.drop_duplicates()
print(f"Paso 2 - Eliminar duplicados: {registros_iniciales - len(df_limpio)} registros acumulados")

# 4.3 Eliminar registros con tempo = 0 o negativo
df_limpio = df_limpio[df_limpio['tempo'] > 0]
print(f"Paso 3 - Eliminar tempo <= 0: {registros_iniciales - len(df_limpio)} registros acumulados")

# 4.4 Eliminar registros con duration_ms = 0 o negativo
df_limpio = df_limpio[df_limpio['duration_ms'] > 0]
print(f"Paso 4 - Eliminar duration_ms <= 0: {registros_iniciales - len(df_limpio)} registros acumulados")

# 4.5 Eliminar registros con mode invalido (diferente de 0 o 1)
df_limpio = df_limpio[(df_limpio['mode'] == 0) | (df_limpio['mode'] == 1)]
print(f"Paso 5 - Eliminar mode invalido: {registros_iniciales - len(df_limpio)} registros acumulados")

# 4.6 Eliminar registros fuera de rango [0,1] para variables normalizadas
for col in ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'liveness', 'speechiness']:
    df_limpio = df_limpio[(df_limpio[col] >= 0) & (df_limpio[col] <= 1)]

print(f"Paso 6 - Eliminar valores fuera de rango [0,1]: {registros_iniciales - len(df_limpio)} registros acumulados")

# ============================================
# 5. TRATAMIENTO DE OUTLIERS (Transformacion logaritmica)
# ============================================
print("\n" + "="*50)
print("TRATAMIENTO DE OUTLIERS")
print("="*50)

# Variables con distribucion asimetrica que requieren transformacion
variables_asimetricas = ['instrumentalness', 'speechiness', 'liveness']

for col in variables_asimetricas:
    antes = df_limpio[col].describe()
    # Transformacion logaritmica log(x+1)
    df_limpio[col] = np.log1p(df_limpio[col])
    despues = df_limpio[col].describe()
    print(f"  {col}: aplicada transformacion log(x+1)")

# ============================================
# 6. EXPORTAR DATOS PROCESADOS
# ============================================
print("\n" + "="*50)
print("EXPORTANDO DATOS PROCESADOS")
print("="*50)

# Dataset completo (todas las columnas) despues de limpieza
df_limpio.to_csv(os.path.join(RUTA_PROCESSED, "universal_top_spotify_songs_clean.csv"), index=False)
print("Exportado: universal_top_spotify_songs_clean.csv")

# Solo las features acusticas (para clustering)
df_features = df_limpio[FEATURES_ACUSTICAS]
df_features.to_csv(os.path.join(RUTA_PROCESSED, "spotify_features_clean.csv"), index=False)
print("Exportado: spotify_features_clean.csv")

# ============================================
# 7. RESUMEN FINAL
# ============================================
print("\n" + "="*50)
print("RESUMEN DEL PREPROCESAMIENTO")
print("="*50)

print(f"""
Dataset original:          {registros_iniciales:,} registros
Dataset despues limpieza:  {len(df_limpio):,} registros
Registros eliminados:      {registros_iniciales - len(df_limpio):,}

Features acusticas seleccionadas: {len(FEATURES_ACUSTICAS)}
Transformaciones aplicadas: log(x+1) para instrumentalness, speechiness, liveness

Archivos generados:
  - universal_top_spotify_songs_clean.csv (dataset completo limpio)
  - spotify_features_clean.csv (solo las {len(FEATURES_ACUSTICAS)} features)
""")

# ============================================
# 8. VERIFICACION FINAL
# ============================================
print("\nVERIFICACION FINAL DE DATOS LIMPIOS:")
print("-"*40)
print(f"Registros totales: {len(df_limpio):,}")
print(f"Valores nulos restantes: {df_limpio[FEATURES_ACUSTICAS].isnull().sum().sum()}")
print(f"Duplicados restantes: {df_limpio.duplicated().sum()}")

print("\nEstadisticas descriptivas de las features limpias:")
print(df_features.describe().round(4))

Cargando dataset...
Dataset cargado. Dimensiones: (2110316, 25)
Columnas totales: 25
Variables seleccionadas para clustering:
  1. danceability
  2. energy
  3. valence
  4. acousticness
  5. instrumentalness
  6. liveness
  7. speechiness
  8. tempo
  9. loudness
  10. duration_ms
  11. mode

DIAGNOSTICO DE CALIDAD
Valores nulos en features acusticas: 0
Registros duplicados (filas identicas): 0

Verificacion de rangos:
  tempo: 1 valores fuera de rango
  loudness: 2108884 valores fuera de rango
  duration_ms: 29 valores fuera de rango

APLICANDO LIMPIEZA
Paso 1 - Eliminar nulos: 0 registros eliminados
Paso 2 - Eliminar duplicados: 0 registros acumulados
Paso 3 - Eliminar tempo <= 0: 1 registros acumulados
Paso 4 - Eliminar duration_ms <= 0: 30 registros acumulados
Paso 5 - Eliminar mode invalido: 30 registros acumulados
Paso 6 - Eliminar valores fuera de rango [0,1]: 30 registros acumulados

TRATAMIENTO DE OUTLIERS
  instrumentalness: aplicada transformacion log(x+1)
  speechiness: ap